# Notebook 01 — Embeddings, By Hand

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com → **File > Upload notebook** → choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths too.

This notebook uses only Python's built-in `math` — there is nothing to install.

**The promise:** by the end, you can explain — to a friend, in 90 seconds, with no jargon —
what an embedding is.

We do not call any model or API here. We **build** tiny embeddings by hand for ten words,
write the similarity math ourselves, and watch the numbers match our intuition. The real
model comes in Notebook 03.

## The central analogy: a map of meaning

> Think of an embedding as **coordinates on a map of meaning.**
>
> On a real map, cities with similar latitude/longitude are close together. On a map of
> meaning, things with similar *meaning* are close together. `cat` sits near `dog`, not
> near `car`.
>
> The twist: this map has **more than two directions**. The real model we use later has
> **384** of them. They aren't labelled — the model invented them. But the *idea* is
> exactly the same as a 2-D map.

**Today we cheat:** we make a map with only **4 directions**, and we label them ourselves.
That's not how real embeddings work, but it shows the *shape* of how they work, in code you
can read top to bottom.

## Step 1 — One import

We use only `math` from Python's standard library. No frameworks today, on purpose, so
every line is readable.

In [ ]:
import math

print("Ready — using only Python's built-in math today.")

## Step 2 — Build 4-D word vectors by hand

We'll place **ten words** in a 4-dimensional space. We pick the four directions by hand and
**name** them, so each vector is readable:

| Direction | Meaning | 0 means… | 1 means… |
|---|---|---|---|
| `animal_ness` | how animal-like | not an animal | definitely an animal |
| `vehicle_ness` | how vehicle-like | not a vehicle | definitely a vehicle |
| `size` | physical size | tiny | huge |
| `alive` | alive / moves on its own | inert | lively |

So `cat` scores high on `animal_ness`, low on `vehicle_ness`, small on `size`, high on
`alive`: `[0.9, 0.0, 0.2, 0.9]`. **Notice:** *someone* makes these numbers up. That's the
point — it shows where the numbers come from before a model hides that from us.

In [ ]:
# Ten words, each a hand-built 4-D vector.
#                       animal  vehicle  size   alive
word_vectors = {
    "cat":      [0.9,   0.0,   0.2,   0.9],
    "dog":      [0.9,   0.0,   0.3,   0.9],
    "lion":     [0.9,   0.0,   0.8,   0.9],
    "mouse":    [0.9,   0.0,   0.1,   0.9],
    "car":      [0.0,   0.9,   0.5,   0.7],
    "truck":    [0.0,   0.9,   0.9,   0.7],
    "bicycle":  [0.0,   0.8,   0.2,   0.5],
    "airplane": [0.0,   0.9,   1.0,   0.8],
    "tree":     [0.0,   0.0,   0.7,   0.6],
    "rock":     [0.0,   0.0,   0.5,   0.0],
}

# Print them in a neat table.
print(f"{'word':<10} {'animal':>8} {'vehicle':>8} {'size':>6} {'alive':>6}")
print("-" * 42)
for word, vec in word_vectors.items():
    print(f"{word:<10} {vec[0]:>8.1f} {vec[1]:>8.1f} {vec[2]:>6.1f} {vec[3]:>6.1f}")

print(f"\n{len(word_vectors)} words placed in a 4-D space.")

### Think about it

- `airplane` has `alive = 0.8`, not 0. Why might the table-builder do that? (Planes *move*
  and feel lively, even though they aren't alive.)
- `bicycle` has `vehicle_ness = 0.8`, not 0.9 like the car. Whoever builds the table
  decides. **The same word can be "close to a car" or "close to a horse" depending on what
  the directions mean.**

## Step 3 — Write the similarity math ourselves

Two words are "similar" if their vectors are similar. But what does that mean with numbers?
Two answers show up everywhere in embeddings.

### Dot product — "how much do these two agree?"

Multiply each matching pair of numbers, then add it all up. Big when both vectors are big
in the same directions.

$$\text{dot}(a, b) = \sum_i a_i \times b_i$$

### Cosine similarity — "are these two pointing the same way?"

Divide the dot product by the lengths of both vectors. This **cancels out length** — only
the *direction* (the angle between the arrows) matters. Cosine runs from -1 to +1; for our
all-positive vectors it stays in 0 to 1.

$$\cos(a, b) = \frac{\text{dot}(a, b)}{|a| \times |b|}$$

> Dot product: "cosine, but longer vectors win." Cosine: "ignore length, just compare
> direction." **For text, cosine is almost always what you want** — and it's the function we
> reuse for the rest of the course.

In [ ]:
# Dot product: multiply matching numbers, then sum them.
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

# Magnitude: the straight-line length of a vector.
def magnitude(v):
    return math.sqrt(sum(x * x for x in v))

# Cosine similarity: dot divided by both lengths — keeps direction, drops length.
def cosine_similarity(a, b):
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

# A vector compared with itself should score exactly 1.0.
cat = word_vectors["cat"]
print(f"cosine(cat, cat) = {cosine_similarity(cat, cat):.3f}   (should be 1.000)")
print("\nSimilarity functions defined.")

## Step 4 — Predict, then verify

Before running the next cell, guess for each pair: will cosine be **high** (≥ 0.95),
**medium** (0.7–0.95), or **low** (< 0.7)?

| Pair | Your guess |
|---|---|
| `cat` ↔ `dog` | ? |
| `cat` ↔ `lion` | ? |
| `car` ↔ `truck` | ? |
| `cat` ↔ `car` | ? |
| `tree` ↔ `rock` | ? |
| `airplane` ↔ `bicycle` | ? |
| `mouse` ↔ `airplane` | ? |

Now run it and see how you did.

In [ ]:
pairs = [
    ("cat",      "dog"),
    ("cat",      "lion"),
    ("car",      "truck"),
    ("cat",      "car"),
    ("tree",     "rock"),
    ("airplane", "bicycle"),
    ("mouse",    "airplane"),
]

print(f"{'word A':<10} {'word B':<10} {'dot':>8}  {'cosine':>8}")
print("-" * 40)
for a, b in pairs:
    va, vb = word_vectors[a], word_vectors[b]
    print(f"{a:<10} {b:<10} {dot_product(va, vb):>8.3f}  {cosine_similarity(va, vb):>8.3f}")

print("\nCompared.")

### What to notice
1. How many did you get right? (Few people get all seven first try.)
2. Compare the `dot` and `cosine` columns. They often agree — but dot punishes short
   vectors, while cosine ignores length. Find a row where they disagree and see why.

## Step 5 — Nearest neighbours: "who hangs out with whom?"

Now flip the question. Instead of "how close are A and B?", ask: *for a given word, which
words are closest?* This is exactly what a vector database does — the heart of search.
We'll write it in five lines.

In [ ]:
# For any word, list its k closest neighbours by cosine similarity.
def nearest(word, k=3):
    others = [w for w in word_vectors if w != word]
    scored = [(w, cosine_similarity(word_vectors[word], word_vectors[w])) for w in others]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:k]

print("Top-3 closest words by cosine similarity:\n")
for word in ["cat", "car", "tree"]:
    neighbours = ", ".join(f"{w} ({s:.2f})" for w, s in nearest(word))
    print(f"  {word:<8} -> {neighbours}")

print("\nNearest-neighbour search in five lines.")

### What just happened

`cat`'s neighbours are other small animals. `car`'s are other vehicles. And `tree`? Its
nearest neighbour turns out to be `airplane` — surprising, until you remember we made both
fairly *large*. A reminder that the directions *we chose* decide what "similar" means.
**This is search in miniature.** Notebook 03 does the same thing — but with 384-dimensional
vectors from a real model over real sentences. The mechanism is identical.

## Step 6 — Your turn: add a word

Below we add `whale` with hand-picked numbers (a huge ocean animal), then show its
neighbours. **Try it yourself:** change the four numbers, or add a different word
(`scooter`, `flower`, `helicopter`...). Predict its neighbours first, then run.

In [ ]:
# Add a new word's 4 numbers:  animal  vehicle  size   alive
word_vectors["whale"] = [0.9,    0.0,    1.0,   0.9]

your_word = "whale"
print(f"Nearest neighbours of '{your_word}':")
for w, s in nearest(your_word):
    print(f"  {w:<10} {s:.3f}")

## Recap

- **What is an embedding?** Coordinates on a map of meaning.
- **Why cosine over dot for text?** Cosine ignores length and compares direction.
- **How do you find similar items?** Cosine to each one, sort, take the top k — exactly what
  `nearest()` did.
- **Our 4-D vectors vs a real model's 384-D vectors?** Ours have human-named directions; the
  model's are learned from data and unreadable. The *shape* is the same.

**Next (Notebook 02):** we used 4 directions because we could read them. But how do you
*see* 4 (or 384) directions at once? That's PCA — squashing many directions into a 2-D
picture you can look at.